In [ ]:
import openai
from tabulate import tabulate

class SimpleGPT4oExperiment:
    def __init__(self, api_key: str):
        """Initialize the experiment with OpenAI API key"""
        self.client = openai.OpenAI(api_key=api_key) 
        
        # Fixed hyperparameters for reproducibility
        self.hyperparameters = {
            "model": "gpt-4o",
            "seed": 42,  # Fixed seed for reproducibility
            "temperature": 1.0,
            "max_tokens": 300,
        }
        
        # One inflation question
        self.question = """
What do you expect the rate of inflation to be over the next 12 months? Please give your best guess.
"""
    
    def ask_question(self, temperature: float) -> dict:
        """Ask the question with specified temperature"""
        try:
            response = self.client.chat.completions.create(
                model=self.hyperparameters["model"],
                messages=[{"role": "user", "content": self.question}],
                seed=self.hyperparameters["seed"],
                temperature=temperature,
                max_tokens=self.hyperparameters["max_tokens"]
            )
            
            return {
                "temperature": temperature,
                "response": response.choices[0].message.content,
                "tokens": response.usage.total_tokens,
                "system_fingerprint": response.system_fingerprint
            }
        except Exception as e:
            return {"error": str(e), "temperature": temperature}
    
    def run_reproducibility_test(self, temperatures: list, runs_per_temp: int = 10):
        """Run multiple tests to check reproducibility"""
        all_results = {}
        
        for temp in temperatures:
            print(f"Testing temperature {temp} with {runs_per_temp} runs...")
            
            results = []
            for run in range(runs_per_temp):
                result = self.ask_question(temp)
                result["run"] = run + 1
                results.append(result)
                print(f"  Run {run + 1}/{runs_per_temp} completed")
            
            all_results[temp] = results
        
        return all_results
    
    def analyze_results(self, all_results: dict):
        """Analyze and display results"""
        print("\n" + "="*80)
        print("REPRODUCIBILITY ANALYSIS")
        print("="*80)
        
        for temp, results in all_results.items():
            print(f"\nTemperature: {temp}")
            print("-" * 40)
            
            # Get all responses (excluding errors)
            responses = [r["response"] for r in results if "error" not in r]
            
            if not responses:
                print("  ❌ All runs failed")
                continue
            
            # Check if all responses are identical
            unique_responses = set(responses)
            
            if len(unique_responses) == 1:
                print(f"  ✅ IDENTICAL across all {len(responses)} runs")
            else:
                print(f"  ❌ DIFFERENT: {len(unique_responses)} unique responses out of {len(responses)} runs")
                
                # Show examples of different responses
                for i, unique_resp in enumerate(list(unique_responses)[:3]):
                    preview = unique_resp[:100].replace('\n', ' ')
                    print(f"    Example {i+1}: {preview}...")
            
            # Show system fingerprint consistency
            fingerprints = set([r["system_fingerprint"] for r in results if "error" not in r])
            if len(fingerprints) == 1:
                print(f"  🔍 System fingerprint consistent: {list(fingerprints)[0]}")
            else:
                print(f"  ⚠️  Multiple system fingerprints: {fingerprints}")
    
    def print_all_results(self, all_results: dict):
        """Print ALL results for detailed analysis"""
        print("\n" + "="*100)
        print("ALL RESULTS - DETAILED OUTPUT")
        print("="*100)
        
        for temp, results in all_results.items():
            print(f"\n🌡️  TEMPERATURE: {temp}")
            print("="*60)
            
            for i, result in enumerate(results):
                print(f"\n--- Run {i+1} ---")
                if "error" in result:
                    print(f"❌ ERROR: {result['error']}")
                else:
                    print(f"🔧 Tokens: {result['tokens']}")
                    print(f"🔍 Fingerprint: {result['system_fingerprint']}")
                    print(f"📝 Response:")
                    print(result['response'])
                    print("-" * 40)
    
    def print_summary_table(self, all_results: dict):
        """Print a summary table"""
        print("\n" + "="*80)
        print("SUMMARY TABLE")
        print("="*80)
        
        table_data = []
        for temp, results in all_results.items():
            for i, result in enumerate(results):
                if "error" not in result:
                    response_preview = result["response"][:60] + "..." if len(result["response"]) > 60 else result["response"]
                    table_data.append([
                        temp,
                        i+1,
                        result["tokens"],
                        response_preview,
                        result["system_fingerprint"][:15] + "..."
                    ])
        
        headers = ["Temp", "Run", "Tokens", "Response Preview", "Fingerprint"]
        print(tabulate(table_data, headers=headers, tablefmt="grid", maxcolwidths=[6, 4, 8, 40, 18]))

# Example usage
def main():
    # Initialize experiment (replace with your API key)
    experiment = SimpleGPT4oExperiment(api_key="your-openai-api-key-here")
    
    print("=== GPT-4o Reproducibility Test ===")
    print(f"Question: Inflation probability estimates")
    print(f"Seed: {experiment.hyperparameters['seed']}")
    print(f"Runs per temperature: 10")
    
    # Test different temperatures
    temperatures = [0.0, 1.0]
    
    # Run the experiment
    results = experiment.run_reproducibility_test(temperatures, runs_per_temp=100)
    
    # Analyze and display results
    experiment.analyze_results(results)
    experiment.print_summary_table(results)
    
    # Print ALL detailed results
    experiment.print_all_results(results)

if __name__ == "__main__":
    main()

=== GPT-4o Reproducibility Test ===
Question: Inflation probability estimates
Seed: 42
Runs per temperature: 10
Testing temperature 0.0 with 100 runs...
  Run 1/100 completed
  Run 2/100 completed
  Run 3/100 completed
  Run 4/100 completed
  Run 5/100 completed
  Run 6/100 completed
  Run 7/100 completed
  Run 8/100 completed
  Run 9/100 completed
  Run 10/100 completed
  Run 11/100 completed
  Run 12/100 completed
  Run 13/100 completed
  Run 14/100 completed
  Run 15/100 completed
  Run 16/100 completed
  Run 17/100 completed
  Run 18/100 completed
  Run 19/100 completed
  Run 20/100 completed
  Run 21/100 completed
  Run 22/100 completed
  Run 23/100 completed
  Run 24/100 completed
  Run 25/100 completed
  Run 26/100 completed
  Run 27/100 completed
  Run 28/100 completed
  Run 29/100 completed
  Run 30/100 completed
  Run 31/100 completed
  Run 32/100 completed
  Run 33/100 completed
  Run 34/100 completed
  Run 35/100 completed
  Run 36/100 completed
  Run 37/100 completed
  Run

In [ ]:
import openai
from tabulate import tabulate

class SimpleGPT4oExperiment:
    def __init__(self, api_key: str):
        """Initialize the experiment with OpenAI API key"""
        self.client = openai.OpenAI(api_key=api_key) 
        
        # Fixed hyperparameters for reproducibility
        self.hyperparameters = {
            "model": "gpt-4o",
            "seed": 42,  # Fixed seed for reproducibility
            "temperature": 0.7,
            "max_tokens": 300,
        }
        
        # One inflation question
        self.question = """
Please estimate the probability (as a percentage) for each of the following inflation/deflation scenarios over the next 12 months.
Each probability must be between 0% and 100%.
You may use up to 2 decimal points (e.g., 7.25%).
The sum of all probabilities must equal exactly 100%.
Return only a list of the numbers (i.e., 50 instead of '50%').

Inflation of 12% or more: _____%
Inflation between 8% and 12%: _____%
Inflation between 4% and 8%: _____%
Inflation between 2% and 4%: _____%
Inflation between 0% and 2%: _____%
Deflation between 0% and 2%: _____%
Deflation between 2% and 4%: _____%
Deflation between 4% and 8%: _____%
Deflation between 8% and 12%: _____%
Deflation of 12% or more: _____%
"""
    
    def ask_question(self, temperature: float) -> dict:
        """Ask the question with specified temperature"""
        try:
            response = self.client.chat.completions.create(
                model=self.hyperparameters["model"],
                messages=[{"role": "user", "content": self.question}],
                seed=self.hyperparameters["seed"],
                temperature=temperature,
                max_tokens=self.hyperparameters["max_tokens"]
            )
            
            return {
                "temperature": temperature,
                "response": response.choices[0].message.content,
                "tokens": response.usage.total_tokens,
                "system_fingerprint": response.system_fingerprint
            }
        except Exception as e:
            return {"error": str(e), "temperature": temperature}
    
    def run_reproducibility_test(self, temperatures: list, runs_per_temp: int = 10):
        """Run multiple tests to check reproducibility"""
        all_results = {}
        
        for temp in temperatures:
            print(f"Testing temperature {temp} with {runs_per_temp} runs...")
            
            results = []
            for run in range(runs_per_temp):
                result = self.ask_question(temp)
                result["run"] = run + 1
                results.append(result)
                print(f"  Run {run + 1}/{runs_per_temp} completed")
            
            all_results[temp] = results
        
        return all_results
    
    def analyze_results(self, all_results: dict):
        """Analyze and display results"""
        print("\n" + "="*80)
        print("REPRODUCIBILITY ANALYSIS")
        print("="*80)
        
        for temp, results in all_results.items():
            print(f"\nTemperature: {temp}")
            print("-" * 40)
            
            # Get all responses (excluding errors)
            responses = [r["response"] for r in results if "error" not in r]
            
            if not responses:
                print("  ❌ All runs failed")
                continue
            
            # Check if all responses are identical
            unique_responses = set(responses)
            
            if len(unique_responses) == 1:
                print(f"  ✅ IDENTICAL across all {len(responses)} runs")
            else:
                print(f"  ❌ DIFFERENT: {len(unique_responses)} unique responses out of {len(responses)} runs")
                
                # Show examples of different responses
                for i, unique_resp in enumerate(list(unique_responses)[:3]):
                    preview = unique_resp[:100].replace('\n', ' ')
                    print(f"    Example {i+1}: {preview}...")
            
            # Show system fingerprint consistency
            fingerprints = set([r["system_fingerprint"] for r in results if "error" not in r])
            if len(fingerprints) == 1:
                print(f"  🔍 System fingerprint consistent: {list(fingerprints)[0]}")
            else:
                print(f"  ⚠️  Multiple system fingerprints: {fingerprints}")
    
    def print_all_results(self, all_results: dict):
        """Print ALL results for detailed analysis"""
        print("\n" + "="*100)
        print("ALL RESULTS - DETAILED OUTPUT")
        print("="*100)
        
        for temp, results in all_results.items():
            print(f"\n🌡️  TEMPERATURE: {temp}")
            print("="*60)
            
            for i, result in enumerate(results):
                print(f"\n--- Run {i+1} ---")
                if "error" in result:
                    print(f"❌ ERROR: {result['error']}")
                else:
                    print(f"🔧 Tokens: {result['tokens']}")
                    print(f"🔍 Fingerprint: {result['system_fingerprint']}")
                    print(f"📝 Response:")
                    print(result['response'])
                    print("-" * 40)
    
    def print_summary_table(self, all_results: dict):
        """Print a summary table"""
        print("\n" + "="*80)
        print("SUMMARY TABLE")
        print("="*80)
        
        table_data = []
        for temp, results in all_results.items():
            for i, result in enumerate(results):
                if "error" not in result:
                    response_preview = result["response"][:60] + "..." if len(result["response"]) > 60 else result["response"]
                    table_data.append([
                        temp,
                        i+1,
                        result["tokens"],
                        response_preview,
                        result["system_fingerprint"][:15] + "..."
                    ])
        
        headers = ["Temp", "Run", "Tokens", "Response Preview", "Fingerprint"]
        print(tabulate(table_data, headers=headers, tablefmt="grid", maxcolwidths=[6, 4, 8, 40, 18]))

# Example usage
def main():
    # Initialize experiment (replace with your API key)
    experiment = SimpleGPT4oExperiment(api_key="your-openai-api-key-here")
    
    print("=== GPT-4o Reproducibility Test ===")
    print(f"Question: Inflation probability estimates")
    print(f"Seed: {experiment.hyperparameters['seed']}")
    print(f"Runs per temperature: 10")
    
    # Test different temperatures
    temperatures = [0.0, 1.0]
    
    # Run the experiment
    results = experiment.run_reproducibility_test(temperatures, runs_per_temp=100)
    
    # Analyze and display results
    experiment.analyze_results(results)
    experiment.print_summary_table(results)
    
    # Print ALL detailed results
    experiment.print_all_results(results)

if __name__ == "__main__":
    main()

=== GPT-4o Reproducibility Test ===
Question: Inflation probability estimates
Seed: 42
Runs per temperature: 10
Testing temperature 0.0 with 100 runs...
  Run 1/100 completed
  Run 2/100 completed
  Run 3/100 completed
  Run 4/100 completed
  Run 5/100 completed
  Run 6/100 completed
  Run 7/100 completed
  Run 8/100 completed
  Run 9/100 completed
  Run 10/100 completed
  Run 11/100 completed
  Run 12/100 completed
  Run 13/100 completed
  Run 14/100 completed
  Run 15/100 completed
  Run 16/100 completed
  Run 17/100 completed
  Run 18/100 completed
  Run 19/100 completed
  Run 20/100 completed
  Run 21/100 completed
  Run 22/100 completed
  Run 23/100 completed
  Run 24/100 completed
  Run 25/100 completed
  Run 26/100 completed
  Run 27/100 completed
  Run 28/100 completed
  Run 29/100 completed
  Run 30/100 completed
  Run 31/100 completed
  Run 32/100 completed
  Run 33/100 completed
  Run 34/100 completed
  Run 35/100 completed
  Run 36/100 completed
  Run 37/100 completed
  Run